In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

# Lendo o arquivo XLSX de um volume do Unity Catalog ou DBFS
df = pd.read_excel("/Volumes/workspace/default/agentes_ia/base_teste.xlsx", engine="openpyxl")
display(df)

In [0]:
%pip install nltk

In [0]:
from nltk.downloader import download
download('stopwords')
from nltk.corpus import stopwords

In [0]:
# Criar um TF-IDF com n-grams de 1 a N

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

In [0]:
# Definindo as stopwords em português
stop_words_portuguese = list(stopwords.words('portuguese'))

In [0]:
# Criação do vetor TF-IDF
Tfidf = TfidfVectorizer(ngram_range=(1, 4),
                      stop_words=stop_words_portuguese, #Já remove as stopwords
                      strip_accents = 'unicode', #Já remove os acentos do texto
                      max_features=15, #Pega as n palavras mais frequentes e corta o número de features
                      norm = 'l1' 
                      #l1 soma do vetor de cada frase é 1;
                      #l2 soma do quadrado dos elementos do vetor é 1; None sem normalizar
                        )
# Corpus
corpus = df['DESC_ITEM_NF'].values

In [0]:
# Transformação das corpus em matriz TF-IDF + transformação em array
Tfidf_X = Tfidf.fit_transform(corpus).toarray()

In [0]:
# Salvar no dataframe
df['vetores_tfidf'] = list(Tfidf_X)
df.head()


O objetivo é agrupar os itens de nota fiscal em grupos, identificar o tema desses grupos para posterior utilização

In [0]:
#Importar kmeans e elbow e silhouette
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [0]:
N_CLUSTERS = 20 # Numero de clusters máximo
CLUSTERS_LIST = list(range(2, N_CLUSTERS + 1))  # Número de clusters a serem testados

In [0]:
# Features
X = df['vetores_tfidf'].tolist()  # Lista de vetores TF-IDF

In [0]:
# Variáveis de armazenamento para os resultados
inertia_values = []
silhouette_scores = []
models = []

In [0]:
# Loop para testar diferentes números de clusters
for n_clusters in CLUSTERS_LIST:
    # Mostrar progresso
    print(f"Testando {n_clusters} clusters...")
    # Criando o modelo KMeans
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    
    # Ajustando o modelo aos dados
    kmeans.fit(X)
    
    # Armazenando os resultados
    inertia_values.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, kmeans.labels_))
    models.append(kmeans)

Visualização

In [0]:
# Plotar os resultados
import matplotlib.pyplot as plt

# Plotar inercia e silhueta
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de inércia
axes[0].plot(CLUSTERS_LIST, inertia_values, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de Clusters', fontsize=12)
axes[0].set_ylabel('Inércia', fontsize=12)
axes[0].set_title('Método do Cotovelo (Elbow Method)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Gráfico de silhueta
axes[1].plot(CLUSTERS_LIST, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Número de Clusters', fontsize=12)
axes[1].set_ylabel('Silhueta Score', fontsize=12)
axes[1].set_title('Score de Silhueta por Número de Clusters', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import silhouette_samples, silhouette_score

# Modelos que serão avaliados
N_CLUSTERS_TO_PLOT = [7,10,15]

# Configuração da figura
fig, axes = plt.subplots(
    1, 3,
    figsize=(20, 7),
    sharex=True,
    sharey=True
)

# Paleta de cores
cmap = plt.cm.nipy_spectral

for ax, n_clusters in zip(axes, N_CLUSTERS_TO_PLOT):

    # Recuperar modelo correspondente
    idx = CLUSTERS_LIST.index(n_clusters)
    model = models[idx]

    # Labels dos documentos
    cluster_labels = model.labels_

    # Calcular silhueta individual
    sample_silhouette_values = silhouette_samples(
        X,
        cluster_labels
    )

    # Silhueta média
    avg_silhouette = silhouette_score(
        X,
        cluster_labels
    )

    y_lower = 10

    for i in range(n_clusters):

        # Silhuetas dos documentos do cluster
        cluster_silhouette_values = sample_silhouette_values[
            cluster_labels == i
        ]

        # Ordenar valores
        cluster_silhouette_values.sort()

        size_cluster_i = cluster_silhouette_values.shape[0]

        y_upper = y_lower + size_cluster_i

        color = cmap(float(i) / n_clusters)

        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7
        )

        # Identificação do cluster
        ax.text(
            -0.05,
            y_lower + 0.5 * size_cluster_i,
            str(i),
            fontsize=9
        )

        y_lower = y_upper + 10

    # Linha da silhueta média
    ax.axvline(
        x=avg_silhouette,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=f"Média: {avg_silhouette:.3f}"
    )

    ax.set_title(
        f"KMeans - {n_clusters} Clusters\n"
        f"Silhouette Score: {avg_silhouette:.4f}"
    )

    ax.set_xlabel("Coeficiente de Silhueta")

    ax.set_xlim([-0.1, 1.0])

    ax.set_yticks([])

    ax.legend(loc="upper right")

    ax.grid(axis="x", alpha=0.3)

axes[0].set_ylabel("Cluster")

fig.suptitle(
    "Comparação dos Gráficos de Silhueta - KMeans",
    fontsize=15,
    y=1.02
)

plt.tight_layout()

plt.savefig(
    "comparacao_silhouette_9_10_11.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [0]:
model = models[5] # Pegar o modelo com 7 clusters
model

In [0]:
df['cluster'] = model.predict(df['vetores_tfidf'].tolist())
df.head()

In [0]:
df['tokens'] = df['DESC_ITEM_NF'].apply( lambda x: x.split())
df.head()

In [0]:
[word for token in df['tokens'].values for word in token][:10]

In [0]:
# Precisamos dar um nome para cada cluster: usaremos a contagem de palavras por cluster para isso
from nltk.util import ngrams
from collections import Counter

# variavel armazenadora
resultados = {
    'cluster': [],
    'tokens_frequentes': [],
    'contagem_tokens': []
}

clusters_unicos = df['cluster'].unique() #Lista de clusters unicos
n = 1 #n-gramas desejado

for c in clusters_unicos:    

    df_filtrado = df[df['cluster'] == c]
    
    # todas as palavras do corpus do cluster c
    all_words = [word for token in df_filtrado['tokens'].values for word in token if word not in stop_words_portuguese] 

    # Criar n-gramas
    ngram_list = list(ngrams(all_words, n))

    # Contar a frequência dos n-gramas
    ngram_freq = Counter(ngram_list)

    # Preparar dados para o gráfico
    n_gramas = [" ".join(ng) for ng, freq in ngram_freq.most_common(5)]
    freq = [freq for ng, freq in ngram_freq.most_common(5)]
    
    # Salvar
    resultados['cluster'].append(c)
    resultados['tokens_frequentes'].append(n_gramas)
    resultados['contagem_tokens'].append(freq)

df_clusters = pd.DataFrame(resultados)
df_clusters.sort_values(by='cluster', ascending=True)

In [0]:
# Criar dicionário com nomes dos clusters baseado nos tokens frequentes
dicio = {
    0: "Diesel Aditivado",  
    1: "Diesel Comum",         
    2: "Gasolina Comum",                
    3: "Diesel s10",          
    4: "Aditivos",          
    5: "Alcool Comum",         
    6: "Conveniência"               
}

df['target'] = df['cluster'].map(dicio)
df.head()

diesel s500 foi classificado como aditivado, mas é comum;

arla 32 é aditivo e foi classificado como aditivo;
